In [1]:
"""
Build assumption_master sheet: fee_structure
-----
Extracts fee % structure per product from the "GLA Fee Structure" block
at rows 4-16, cols G-J of the Assumptions sheet.

Schema: product | fee_type | fee_value | fee_unit | param_type | notes
fee_type    ∈ {pct_of_net_premium, rand_fee_recurring, aids_pct_of_gla}
fee_unit    ∈ {pct, rand, pct_of_gla}
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib, csv, openpyxl, polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE_XLSX = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG = SNAPSHOT_DIR / "build_log.csv"

SHEET_NAME = "fee_structure"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")

Building: fee_structure


In [2]:
def get_col_letter(col):
    return openpyxl.utils.get_column_letter(col)

# From earlier diagnostic:
# G4=Product, H4='% of NET Premium', I4='Rand Fee Recurring', J4='Stand-Alone LSDB Aids % of GLA'
# G5=GLA, G6=SP Life, G7=CI, G8=LSDB, G9=Dismemberment, ...

print("GLA Fee Structure block (rows 4-17, cols G-J):")
print("-" * 90)
for row in range(4, 18):
    g = ws.cell(row=row, column=7).value   # G = product label
    h = ws.cell(row=row, column=8).value   # H = % of NET Premium
    i = ws.cell(row=row, column=9).value   # I = Rand Fee Recurring
    j = ws.cell(row=row, column=10).value  # J = Stand-Alone LSDB AIDS % of GLA
    print(f"  Row {row:2d}: G={g!r:35s} H={h!r:10s} I={i!r:10s} J={j!r}")


GLA Fee Structure block (rows 4-17, cols G-J):
------------------------------------------------------------------------------------------
  Row  4: G='Product'                           H='% of NET Premium' I='Rand Fee Recurring' J='Stand-Alone LSDB Aids % of GLA'
  Row  5: G='GLA'                               H=0.06       I=0          J=0.6
  Row  6: G='SP Life'                           H=0.06       I=0          J=None
  Row  7: G='CI'                                H=0.075      I=0          J='Main Product'
  Row  8: G='LSDB'                              H=0.075      I=0          J='GLA'
  Row  9: G='Dismemberment'                     H=0.075      I=0          J='Main Fee PC'
  Row 10: G='Standalone Critical Inc'           H=0.075      I=0          J=0.06
  Row 11: G='Acc Death'                         H=0.06       I=0          J='Main Rand Amt'
  Row 12: G='Acc DB'                            H=0.075      I=0          J=0
  Row 13: G='SP DB'                             H=0.075     